# xHuBERT Experiment 2+7: HuBERT Frozen Baseline + Layer Probing
**De tai**: He thong goi y san pham dua tren phan tich giong noi va cam xuc
**Hoc vien**: Nguyen Tan Nhu | **GVHD**: TS. Bui Thanh Hung (IUH)

**Yeu cau**: `Runtime` -> `Change runtime type` -> **T4 GPU** -> Save

## Pipeline
```
Step 0: Setup
Step 1: Exp2 — HuBERT frozen + SVM/RF (5-fold + LOSGO)
Step 2: Exp7 — Per-layer probing with logistic regression (LOSGO)
```

In [ ]:
# Kiem tra GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), "GB")
else:
    print("WARNING: GPU not enabled! Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/xhubert_results/"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

In [ ]:
# Install dependencies (Colab has torch, numpy, sklearn, matplotlib)
!pip install -q transformers==4.51.3 librosa huggingface-hub safetensors tqdm seaborn

# Verify versions
import torch, transformers, librosa
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"librosa:      {librosa.__version__}")

In [ ]:
# Upload .py modules to Colab
# Option 1: Upload manually via Files panel (drag & drop)
# Option 2: Clone from repo
# !git clone https://github.com/nhunet/xhubert-experiments.git
# %cd xhubert-experiments

# Verify required files
import os
required = [
    "config.py", "data.py", "features.py", "protocols.py",
    "stats.py", "utils.py",
    "models/__init__.py", "models/ml_classifiers.py",
    "models/xhubert.py", "models/hubert_vanilla.py",
    "models/fusion.py", "models/dl_1d.py", "models/dl_2d.py",
]
for f in required:
    status = "OK" if os.path.exists(f) else "MISSING"
    print(f"  [{status}]  {f}")

In [ ]:
# Download RAVDESS dataset
import os
RAVDESS_PATH = "./RAVDESS"
if not os.path.exists(RAVDESS_PATH):
    print("Downloading RAVDESS ...")
    !wget -q https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip
    !unzip -q Audio_Speech_Actors_01-24.zip -d RAVDESS/
    print("Done!")
else:
    print(f"RAVDESS already exists at {RAVDESS_PATH}")
    !find {RAVDESS_PATH} -name "*.wav" | wc -l

In [ ]:
# Set save directory
import config
config.SAVE_DIR = SAVE_DIR
config.CKPT_DIR = os.path.join(SAVE_DIR, "checkpoints")
os.makedirs(config.CKPT_DIR, exist_ok=True)
print(f"Results -> {config.SAVE_DIR}")
print(f"Checkpoints -> {config.CKPT_DIR}")

### Keep Colab Alive
Paste this into your **browser Console** (F12 -> Console) to prevent idle timeout:
```javascript
function ClickConnect() {
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
```

## Load Dataset (16kHz for HuBERT)

In [ ]:
from data import RavdessDataset
import config

dataset = RavdessDataset(sr=config.SR_HUBERT)
dataset.print_summary()

## Exp2: HuBERT Frozen Baseline

In [ ]:
from experiments.exp2_hubert_frozen import run_exp2

df_exp2 = run_exp2(dataset=dataset, force=False)
print(df_exp2.groupby(["Model", "Protocol"])["accuracy"].agg(["mean", "std"]).round(2))

## Exp7: Layer Probing

In [ ]:
from experiments.exp7_layer_probing import run_exp7

df_exp7 = run_exp7(dataset=dataset, force=False)
print(df_exp7.groupby("Layer")["accuracy"].agg(["mean", "std"]).round(2))

## Visualization

In [ ]:
from visualization.plots import plot_exp7_layer_curve
import numpy as np, os

# Load alpha weights if available (from Exp3)
alpha_path = os.path.join(config.SAVE_DIR, "layer_weights_fold0_seed42.npy")
alpha = np.load(alpha_path) if os.path.exists(alpha_path) else None

plot_exp7_layer_curve(df_exp7, alpha_weights=alpha)
print("Layer probing figure saved!")